In [ ]:
import csv
import torch
from sentence_transformers import SentenceTransformer
import sys
sys.path.append("/workspaces/hallucilation_in_llm")
from model.hf_model import HFModel

from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics



prompts_tr = [
    "Fransa'nın başkenti neresidir?",
    "Fotosentezi açıklayın.",
    "Mona Lisa'yı kim yaptı?",
    "Kara delikleri tanımlayın.",
    "Kuantum dolanıklığı nedir?"
]

ground_truth_tr = [
    "Paris",
    "Fotosentez, bitkilerin ışığı kimyasal enerjiye dönüştürdüğü süreçtir.",
    "Leonardo da Vinci",
    "Kara delik, hiçbir şeyin kaçamadığı kadar güçlü yerçekimine sahip uzay bölgesidir.",
    "Kuantum dolanıklığı, parçacıkların mesafeye bakılmaksızın birbirleriyle ilişkili olduğu bir fenomendir."
]



model = HFModel("gpt2")  # Örnek: yerel GPT2

uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
    "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
    "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="tr")
}

final_score_calc = FinalScore()
evaluator = Evaluator(final_score_calc)
decider = HallucinationDecider(thresholds={"hallucination": 0.7})

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)


results = []

for prompt in prompts_tr:
    res = pipeline.run(prompt)
    results.append(res)
    print(f"Prompt: {prompt}")
    print(f"Responses: {res['responses']}")
    print(f"Uncertainty metrics: {res['uncertainty']}")
    print(f"Final evaluation: {res['evaluation']}")
    print(f"Decision: {res['decision']}")
    print("="*50)


class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def compute_similarity(self, responses):
        sim_scores = []
        for gt_text, resp_list in zip(self.ground_truth, responses):
            gt_emb = self.model.encode(gt_text, convert_to_tensor=True, normalize_embeddings=True)
            resp_embs = self.model.encode(resp_list, convert_to_tensor=True, normalize_embeddings=True)
            sims = torch.nn.functional.cosine_similarity(resp_embs, gt_emb.unsqueeze(0))
            sim_scores.append(float(sims.mean()))
        return sim_scores

responses_list = [r['responses'] for r in results]
gt_eval = GroundTruthSemanticEvaluator(ground_truth_tr)
similarity_scores = gt_eval.compute_similarity(responses_list)

final_scores = [r['evaluation']['final_score'] for r in results]

# Pearson ve Spearman
pearson, spearman = StatisticalAnalyzer.correlation(final_scores, similarity_scores)

# AUROC ve PR-AUC (StatisticalAnalyzer sınıfı içinde binary dönüşüm yapıyor)
auroc = StatisticalAnalyzer.auroc(final_scores, similarity_scores, threshold=0.5)
pr_auc = StatisticalAnalyzer.pr_auc(final_scores, similarity_scores, threshold=0.5)

# Brier ve ECE
brier = CalibrationMetrics.brier_score(final_scores, similarity_scores)
ece = CalibrationMetrics.expected_calibration_error(final_scores, similarity_scores)


csv_columns = [
    "Prompt",
    "Responses",
    "Blackbox_Uncertainty",
    "Graybox_Uncertainty",
    "Whitebox_Uncertainty",
    "Semantic_Uncertainty",
    "Final_Score",
    "Decision",
    "Semantic_Similarity",
    "Pearson_Corr",
    "Spearman_Corr",
    "AUROC",
    "PR_AUC",
    "Brier_Score",
    "ECE"
]

csv_file = "pipeline_results_tr.csv"
with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    for i, r in enumerate(results):
        writer.writerow({
            "Prompt": prompts_tr[i],
            "Responses": "; ".join(r['responses']),
            "Blackbox_Uncertainty": r['uncertainty'].get('blackbox', None),
            "Graybox_Uncertainty": r['uncertainty'].get('graybox', None),
            "Whitebox_Uncertainty": r['uncertainty'].get('whitebox', None),
            "Semantic_Uncertainty": r['uncertainty'].get('semantic', None),
            "Final_Score": r['evaluation']['final_score'],
            "Decision": r['decision'],
            "Semantic_Similarity": similarity_scores[i],
            "Pearson_Corr": pearson,
            "Spearman_Corr": spearman,
            "AUROC": auroc,
            "PR_AUC": pr_auc,
            "Brier_Score": brier,
            "ECE": ece
        })

print(f"Tüm sonuçlar '{csv_file}' dosyasına kaydedildi.")


/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 406.71it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loadi

Prompt: Fransa'nın başkenti neresidir?
Responses: ["Fransa'nın başkenti neresidir? (You have arrived and have a seat on a bus.)\n\nNo! (What is that?)\n\nThe train left the station and then stopped in the center of the square.\n\nMuktürküş:\n\n", 'Fransa\'nın başkenti neresidir?\n\n"Sünciğuçarın kışmazın yas müzküniçı. Sünciğur nadırıdın, sünci', "Fransa'nın başkenti neresidir? Eğüyük uğuririn zem çığırül!"]
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.0, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 0.0, 'white_consistency': 0.3333333333333333, 'semantic_consistency': 0.6361387411753336, 'uncertainty': 0.36386125882466636}
Final evaluation: {'metrics': {'entropy': nan, 'confidence': 0.0, 'self_consistency': 0.3333333333333333}, 'final_score': nan}
Decision: reliable


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 721.37it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 721.56it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 720.82it/s, Material

Prompt: Fotosentezi açıklayın.
Responses: ['Fotosentezi açıklayın.\n\nKımıldin - kımıldin.\n\nNisar - nekar.\n\nSomir - somir.\n\nYusılız - yusılı', 'Fotosentezi açıklayın. pic.twitter.com/QdW5dKbLXXa — Вилака Скоружа (@Вилака) January 28, 2016\n\nThe', 'Fotosentezi açıklayın. İbrahim Bekut. Nihat Çşmılıç. Pımıpı, A. & Alakın. 2010. The risk of smoking and related health problems in Turkish men. J']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.05548179955504352, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 1.1576852873580697e-39, 'white_consistency': 0.3333333333333333, 'semantic_consistency': 0.2747424135605494, 'uncertainty': 0.7252575864394506}
Final evaluation: {'metrics': {'entropy': nan, 'confidence': 0.05548179955504352, 'self_consistency': 0.3333333333333333}, 'final_score': nan}
Decision: reliable


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 748.60it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 687.54it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 722.84it/s, Material

...

Prompt: Kara delikleri tanımlayın.
Responses: ['Kara delikleri tanımlayın.\n\nThe second of my two books, The Book of the Dragon, is based on an older book by P. T. S. Janssen and I have been using the same style since that time. It is a wonderful and complex,', 'Kara delikleri tanımlayın.com\n\nThe country\'s two main broadcasters have come under fire for airing cartoons of the Prophet Muhammad on television and in print in Turkey.\n\n"The cartoons were produced by Turkish channel TVN, which broadcast cartoons of Muhammad," a Twitter post', 'Kara delikleri tanımlayın.\n\n"Dinık kara ogli naman darışımı lalıpılılın ogli yinıdın. Nuzı makıyız']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.04157769235366984, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 6.2962717576391004e-46, 'white_consistency': 0.3333333333333333, '

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 688.56it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 728.90it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 747.67it/s, Material

Prompt: Kuantum dolanıklığı nedir?
Responses: ['Kuantum dolanıklığı nedir?\n\nThe letter has been deleted.\n\nCopyright © 2018 The Washington Times, LLC. Click here for reprint permission.', 'Kuantum dolanıklığı nedir? — Огут Кучаний (@kalandisti) November 3, 2017\n\nFinnish broadcaster TASS confirmed Friday that the group is planning to stage a terrorist attack in central', 'Kuantum dolanıklığı nedir?\n\nT. Yıldırımımımımımımımımımımımımımımımımımımımımı']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.0, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 0.0, 'white_consistency': 0.3333333333333333, 'semantic_consistency': 0.24382346651206416, 'uncertainty': 0.7561765334879358}
Final evaluation: {'metrics': {'entropy': nan, 'confidence': 0.0, 'self_consistency': 0.3333333333333333}, 'final_score': nan}
Decision: reliable


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 716.77it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tüm sonuçlar 'pipeline_results_tr.csv' dosyasına kaydedildi.


In [1]:
!pip install sentence_transformers

In [2]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 15.8 MB/s  0:00:20m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 37.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 27.5 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 71.0 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 75.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 27.5 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 71.1 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 76.9 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 55.6 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/28

In [2]:
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [requests]


In [1]:
import csv
import torch
from sentence_transformers import SentenceTransformer
import sys
sys.path.append("/workspaces/hallucilation_in_llm")
from model.hf_model import HFModel

from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics

prompts_en = [
    "What is the capital of France?",
    "Explain photosynthesis.",
    "Who painted the Mona Lisa?",
    "Define black holes.",
    "What is quantum entanglement?"
]

ground_truth_en = [
    "Paris",
    "Photosynthesis is the process by which plants convert light into chemical energy.",
    "Leonardo da Vinci",
    "A black hole is a region in space with gravity so strong that nothing can escape.",
    "Quantum entanglement is a phenomenon where particles remain connected regardless of distance."
]


model = HFModel("EleutherAI/pythia-70m")  # Example: local GPT2

uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
    "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
    "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="en")
}

final_score_calc = FinalScore()
evaluator = Evaluator(final_score_calc)
decider = HallucinationDecider(thresholds={"hallucination": 0.7})

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)


results = []

for prompt in prompts_en:
    res = pipeline.run(prompt)
    results.append(res)
    print(f"Prompt: {prompt}")
    print(f"Responses: {res['responses']}")
    print(f"Uncertainty metrics: {res['uncertainty']}")
    print(f"Final evaluation: {res['evaluation']}")
    print(f"Decision: {res['decision']}")
    print("="*50)


class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def compute_similarity(self, responses):
        sim_scores = []
        for gt_text, resp_list in zip(self.ground_truth, responses):
            gt_emb = self.model.encode(gt_text, convert_to_tensor=True, normalize_embeddings=True)
            resp_embs = self.model.encode(resp_list, convert_to_tensor=True, normalize_embeddings=True)
            sims = torch.nn.functional.cosine_similarity(resp_embs, gt_emb.unsqueeze(0))
            sim_scores.append(float(sims.mean()))
        return sim_scores

responses_list = [r['responses'] for r in results]
gt_eval = GroundTruthSemanticEvaluator(ground_truth_en)
similarity_scores = gt_eval.compute_similarity(responses_list)


final_scores = [r['evaluation']['final_score'] for r in results]

# Pearson and Spearman correlation
pearson, spearman = StatisticalAnalyzer.correlation(final_scores, similarity_scores)

# AUROC and PR-AUC
auroc = StatisticalAnalyzer.auroc(final_scores, similarity_scores, threshold=0.5)
pr_auc = StatisticalAnalyzer.pr_auc(final_scores, similarity_scores, threshold=0.5)

# Brier score and ECE
brier = CalibrationMetrics.brier_score(final_scores, similarity_scores)
ece = CalibrationMetrics.expected_calibration_error(final_scores, similarity_scores)


csv_columns = [
    "Prompt",
    "Responses",
    "Blackbox_Uncertainty",
    "Graybox_Uncertainty",
    "Whitebox_Uncertainty",
    "Semantic_Uncertainty",
    "Final_Score",
    "Decision",
    "Semantic_Similarity",
    "Pearson_Corr",
    "Spearman_Corr",
    "AUROC",
    "PR_AUC",
    "Brier_Score",
    "ECE"
]

csv_file = "pipeline_results_en.csv"
with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    for i, r in enumerate(results):
        writer.writerow({
            "Prompt": prompts_en[i],
            "Responses": "; ".join(r['responses']),
            "Blackbox_Uncertainty": r['uncertainty'].get('blackbox', None),
            "Graybox_Uncertainty": r['uncertainty'].get('graybox', None),
            "Whitebox_Uncertainty": r['uncertainty'].get('whitebox', None),
            "Semantic_Uncertainty": r['uncertainty'].get('semantic', None),
            "Final_Score": r['evaluation']['final_score'],
            "Decision": r['decision'],
            "Semantic_Similarity": similarity_scores[i],
            "Pearson_Corr": pearson,
            "Spearman_Corr": spearman,
            "AUROC": auroc,
            "PR_AUC": pr_auc,
            "Brier_Score": brier,
            "ECE": ece
        })

print(f"All English results saved to '{csv_file}'")


/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 707.97it/s, Materializing param=gpt_neox.layers.5.post_attention_layernorm.weight] 
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|██████████| 103/103 [00:02<00:00, 34.71it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+----

Prompt: What is the capital of France?
Responses: ['What is the capital of France?\n\nThe French capital is a place where the French capital is divided into two parts. The first part is the capital of France, which is divided into two parts, the capital of France, which is divided into two parts, the capital of France', 'What is the capital of France?\n\nThe French capital is a place where the people are not alone. The people are not alone. The people are not alone. The people are not alone. The people are not alone. The people are not alone. The people are not alone', 'What is the capital of France?\n\nThe French are the only ones who can afford to pay their taxes. They are the only ones who can afford to pay their taxes. They are the only ones who can afford to pay their taxes. They are the only ones who can afford']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence'

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 685.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 731.67it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 596.19it/s, Materializing param=poole

Prompt: Explain photosynthesis.
Responses: ['Explain photosynthesis.\n\nThe first step in the process of converting the water to the water is to convert the water to the water. The water is then converted into a water-soluble salt, which is then added to the water. The salt is then added to', 'Explain photosynthesis.\n\nThe first step in the process of the synthesis of the photosynthesis is to use the photosynthesis process to synthesize the photosynthetic pathway. The synthesis of the photosynthesis pathway is then carried out using the photosynthesis process. The synthesis of the', 'Explain photosynthesis.\n\nThe first step in the process is to understand the structure of the protein in the cell, the structure of the cell, and the cell structure of the cell. This is the structure of the cell, and the cell is the structure of the']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gr

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 715.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 622.78it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 691.04it/s, Materializing param=poole

...

Prompt: Define black holes.
Responses: ['Define black holes.\n\nThe black hole is a black hole, with a black hole and a black hole.\n\nThe black hole is a black hole, with a black hole and a black hole.\n\nThe black hole is a black hole, with a', 'Define black holes.\n\nA:\n\nThis is a very simple question.\nThe problem is that the only way to do this is to have a black hole in the background.\nThe only way to do this is to have a black hole in the background', 'Define black holes.\n\nThe first black hole is the first black hole of the universe. The black hole is the first black hole of the universe.\n\nThe black hole is the first black hole of the universe. The black hole is the first black hole of']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.28576938704023036, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 0.00045

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 677.66it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 727.56it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 698.27it/s, Materializing param=poole

Prompt: What is quantum entanglement?
Responses: ['What is quantum entanglement?\n\nThe quantum entanglement is a measure of the number of particles in a system, which is the number of particles that can be measured. The number of particles in a system is the number of particles that can be measured.\n\nThe number of', 'What is quantum entanglement?\n\nA:\n\nThe quantum entanglement is a quantum entanglement of the form $A_1\\otimes\\cdots\\otimes A_n$ which is a quantum state. It is the quantum entanglement of the form $A_1\\otimes', 'What is quantum entanglement?\n\nThe quantum entanglement is a measure of how much a quantum system can be. The quantum system can be described by a quantum system, and it can be described by a quantum system.\n\nThe quantum system can be described by a quantum system']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.24803764137718

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 177.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All English results saved to 'pipeline_results_en.csv'


In [2]:
from stat_metrics import StatisticalAnalyzer

scores = [0.1, 0.2, 0.8, 0.9, 0.4, 0.7]
labels = [0, 0, 1, 1, 0, 1]

pearson, spearman = StatisticalAnalyzer.correlation(scores, labels)
auroc = StatisticalAnalyzer.auroc(scores, labels)
pr_auc = StatisticalAnalyzer.pr_auc(scores, labels)

print("Pearson:", pearson)
print("Spearman:", spearman)
print("AUROC:", auroc)
print("PR-AUC:", pr_auc)


Pearson: 0.9372403389139513
Spearman: 0.87831006565368
AUROC: 1.0
PR-AUC: 1.0
